# 17.14 模型能預先適應量化誤差嗎？


訓練好的模型直接量化，可能對粗刻度很敏感。可以在訓練期間就讓它看見量化後的特徵與權重，學習避開容易受傷的表示嗎？這叫Quantization-Aware Training（QAT，量化感知訓練）。但我們在[梯度暖身](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)知道，學習需要微小變動如何影響誤差；round取整在格子內幾乎不變，直接對它求梯度常得到零，難以教上游權重。

先回顧[刻度捨入](https://birdhackor.github.io/tiny-perceptron-vlm/17.2.html)。輸入 `[0.1,0.4,0.9]`，本工具的四位元對稱量化使用-7至7的整數碼，最大正碼是7，所以把最大絕對值0.9除以7得到scale，還原約 `[0.1286,0.3857,0.9]`。QAT常用fake quantization（模擬量化）：向前確實使用這些還原數字，向後卻用一個近似規則，例如把取整視為「輸入原樣通過」來傳梯度。這叫straight-through estimator（STE，直通估計），不是取整函數真正的導數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.quantization import fake_quantize

x = torch.tensor([0.1, 0.4, 0.9], requires_grad=True)
y = fake_quantize(x, bits=4)
y.sum().backward()
print("向前", y.detach().round(decimals=4).tolist())
print("STE梯度", x.grad.tolist())
hard = x.detach().clone().requires_grad_()
scale = 0.9 / 7
plain = (hard / scale).round() * scale
plain.sum().backward()
print("直接round梯度", hard.grad.tolist())

`fake_quantize`用 `x + (還原值-x).detach()`實作這個示範。向前數值中x相消，剩下還原值；`detach`阻止括號裡的修正項回傳梯度，向後只剩外面直接的x路徑。誤差先取y總和，對各y的導數是1，因此x.grad得到 `[1,1,1]`。建立hard時，`x.detach()`先切斷與原x的求導關係，`.clone()`複製數值到獨立資料，`.requires_grad_()`再把這份副本設為接下來可求導的輸入；它不會接回原x的關係。因此hard可保存自己的梯度，普通round路徑得到 `[0,0,0]`。兩條路的向前捨入規則相同，向後規則刻意不同。

第一行約 `[0.1286,0.3857,0.9]`，比原始輸入有小偏移；第二行並不是說這些偏移的精確導數為1，而是我們選擇了可用於學習的近似。這讓訓練有機會調整權重，減少換成低位元時的任務損失。近似是否有效仍需實驗，不能從「有非零梯度」推論模型一定改善。

模擬量化一般仍保留浮點權重，向前計算（forward）臨時體驗格子誤差，所以這個例子沒有生成較小權重檔，也沒有使用專為低位元資料安排讀取與乘加的底層運算（kernel）。完整QAT流程需要加入適當模擬位置、訓練、轉換成實際打包格式，然後使用轉換版重新檢查任務品質與成本。若模擬規則和最後儲存格式不同，訓練時適應的誤差也可能對不上。

練習只把第一個誤差 `y.sum()` 改成 `y.square().sum()`。先預測STE梯度變成兩倍量化後的y，約 `[0.2571,0.7714,1.8]`，再重跑核對。直接round那條路維持原來的總和誤差，仍是零；這能區分誤差函數提供的學習訊號與模擬量化採用的近似傳遞規則。
